# Notebook 01 — Introducción a PL/pgSQL y bloques anónimos

Primer sub-bloque del Tema 06. **PL/pgSQL** (*Procedural Language for PostgreSQL*) es el lenguaje procedural que vive **dentro del motor**. Te permite escribir lógica de control (`IF`, `CASE`, `FOR`, `WHILE`), declarar variables, y empaquetar lógica en procedimientos almacenados.

En este notebook conoces el lenguaje desde su forma más simple — el **bloque anónimo** `DO $$ ... $$;` — y la **declaración de variables**, antes de pasar a las estructuras de control (Notebook 02) y los procedimientos almacenados (Notebook 03).

> **¿Dónde encaja en el stack de BI?** PL/pgSQL vive **en el servidor PostgreSQL**, no en las herramientas de visualización. Power BI (Power Query) y Metabase **no ejecutan** PL/pgSQL: son clientes que piden un *result set* y lo grafican. Para aprovecharlo desde ellos, la vía es **indirecta** — empaquetas la lógica en una **función que devuelve una tabla** y la llamas con `SELECT * FROM func(...)`. Los bloques `DO`, los procedimientos (`CALL`) y `RAISE NOTICE` se ejecutan desde un cliente SQL de escritorio (DBeaver/psql), no desde la capa de BI.

**Contenido de este notebook:**

- [Setup](#setup)
- [¿Por qué PL/pgSQL?](#por-qué-plpgsql)
- [El bloque anónimo `DO $$ ... $$;`](#el-bloque-anónimo-do----)
- [Declaración y asignación de variables](#declaración-y-asignación-de-variables)
- [Bloques anidados](#bloques-anidados)

## Setup

In [ ]:
# Setup — instala JupySQL si hace falta (Colab trae ipython-sql, no JupySQL).
import importlib.util, subprocess, sys
if importlib.util.find_spec("jupysql") is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "ipython-sql"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"], check=True)
    print("⚠ JupySQL instalado. REINICIA el kernel (Entorno de ejecución → Reiniciar sesión)")
    print("  y vuelve a correr esta celda y las siguientes.")
else:
    print("✓ JupySQL listo.")

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine
# Devuelve cada query como DataFrame de pandas (mejor render en Colab, y
# el resultado es directamente manipulable con pandas).
%config SqlMagic.autopandas = True

## ¿Por qué PL/pgSQL?

SQL puro es **declarativo**: describes qué quieres, el motor decide cómo obtenerlo. Eso es genial para la mayoría de operaciones de BI — un `SELECT` con `GROUP BY` cubre el 80% del trabajo.

Pero hay cosas que SQL puro **no puede expresar directamente**:

- Lógica condicional compleja: "si hay más de X clientes en la región Y, ejecuta el proceso Z; si no, otro distinto".
- Iteración con estado: "para cada categoría, calcula su ranking acumulado y guarda el resultado".
- Manejo de errores: "intenta cargar; si falla por FK, registra el error y continúa con la siguiente fila".
- Empaquetar lógica reutilizable: "esta serie de pasos los necesito desde 5 lugares distintos".

**PL/pgSQL agrega:**

| Capacidad | SQL puro | PL/pgSQL |
|---|---|---|
| Variables locales | ❌ | ✅ `DECLARE` |
| Condicionales | Solo `CASE` (expresión) | ✅ `IF`, `CASE` (sentencia) |
| Loops | ❌ | ✅ `LOOP`, `FOR`, `WHILE` |
| Procedimientos con efectos | ❌ | ✅ `CREATE PROCEDURE` |

> El `IF` como sentencia solo existe **dentro** de PL/pgSQL; en SQL plano solo tienes `CASE`. (Dato: MySQL sí tiene una función `IF(cond, a, b)`, pero PostgreSQL **no** — ahí usas `CASE`.)

**La regla operativa:** quédate en SQL puro siempre que puedas. PL/pgSQL es para cuando SQL puro **realmente no alcanza**. El motor optimiza mucho mejor un `INSERT … SELECT` que un `FOR` loop fila por fila.

## El bloque anónimo `DO $$ ... $$;`

La forma más simple de ejecutar PL/pgSQL: un bloque **anónimo** que corre una vez y se descarta. Sintaxis básica:

```sql
DO $$
BEGIN
    -- código PL/pgSQL aquí
END
$$;
```

Las claves:

- **`DO`** — palabra clave que dice "voy a ejecutar un bloque procedural".
- **`$$ ... $$`** — *dollar-quoted string*. Es la forma de PostgreSQL de delimitar el código sin tener que escapar comillas internas. Todo lo que esté entre los dos `$$` es el cuerpo del bloque.
- **`BEGIN ... END`** — los delimitadores del bloque PL/pgSQL propiamente dicho.
- El `;` final cierra la sentencia `DO`.

**`$$` vs `BEGIN ... END` — son de capas distintas:**

- **`$$ ... $$`** es un **delimitador de string** (nivel SQL): encierra el cuerpo como texto, sin tener que escapar comillas. Es el equivalente a los `"""` de Python — de hecho `DO $$ ... $$` se parece a `exec(""" ... """)`.
- **`BEGIN ... END`** es un **delimitador de bloque de código** (nivel PL/pgSQL): marca dónde empiezan y terminan las instrucciones.

Primero PostgreSQL toma todo lo que está entre `$$...$$` como un string y se lo entrega al intérprete de PL/pgSQL; ese intérprete es el que lee el `BEGIN ... END`.

> ⚠️ El `BEGIN` de PL/pgSQL **no es** el `BEGIN` de transacción de SQL. Mismo nombre, cosas distintas según el contexto.

Un primer ejemplo:

In [ ]:
%%sql
-- Un bloque anónimo no devuelve filas; para ver su resultado desde JupySQL,
-- el bloque escribe en una tabla temporal y luego la consultamos.
DROP TABLE IF EXISTS salida;
CREATE TEMP TABLE salida (mensaje TEXT);

DO $$
BEGIN
    INSERT INTO salida VALUES ('Hola desde PL/pgSQL');
END
$$;

SELECT * FROM salida;

Un bloque `DO` corre en el servidor pero **no devuelve un *result set***. Para *ver* lo que produce desde JupySQL, el patrón es: el bloque **escribe en una tabla temporal** y luego un `SELECT` la muestra.

> En `psql` se suele usar `RAISE NOTICE` para imprimir mensajes, pero ese canal **no se muestra** en JupySQL — por eso aquí usamos la tabla temporal.

### `RAISE NOTICE` en un editor SQL de escritorio

Si en vez de JupySQL usas **DBeaver** o **`psql`**, el mecanismo natural para que un bloque "hable" es `RAISE NOTICE` — ahí **sí** se muestra. Copia y ejecuta esto en tu editor:

```sql
DO $$
DECLARE
    total INTEGER;
BEGIN
    SELECT COUNT(*) INTO total FROM northwind_dwh.dim_customer;
    RAISE NOTICE 'Hay % clientes', total;
END
$$;
```

- Cada `%` del texto es un **placeholder** que se reemplaza, en orden, por los argumentos que van después de la coma (`total` aquí).
- En **DBeaver** el mensaje aparece en la pestaña de **salida / Output** (o el log de la sesión); en **`psql`** sale como una línea `NOTICE:  Hay 91 clientes`.

> En este notebook usamos la tabla temporal porque JupySQL **no muestra** ese canal de mensajes. `RAISE NOTICE` es la opción equivalente cuando trabajas desde un cliente SQL de escritorio.

## Declaración y asignación de variables

Las variables se declaran en una sección **`DECLARE`** que va **antes** del `BEGIN`:

```sql
DO $$
DECLARE
    nombre  TEXT;            -- declarada, valor NULL por default
    edad    INTEGER := 30;   -- declarada e inicializada
    activo  BOOLEAN := TRUE;
BEGIN
    -- código
END
$$;
```

Detalles importantes:

- **Tipos:** cualquier tipo SQL válido (`TEXT`, `NUMERIC(10,2)`, `DATE`, `BOOLEAN`, etc.) más tipos compuestos como `RECORD`, `%ROWTYPE`, `%TYPE`.
- **Asignación:** el operador es **`:=`**, NO `=`. El `=` se usa para comparación.
- **Constantes:** agrega `CONSTANT` antes del tipo: `pi CONSTANT NUMERIC := 3.14159;`.
- **NOT NULL:** puedes forzar que una variable no pueda ser NULL: `nombre TEXT NOT NULL := 'Ana';` (necesita valor inicial).

In [ ]:
%%sql
DROP TABLE IF EXISTS salida;
CREATE TEMP TABLE salida (etiqueta TEXT, valor TEXT);

DO $$
DECLARE
    nombre          TEXT    := 'Aurora';
    version         NUMERIC := 17.4;
    total_clientes  INTEGER;
BEGIN
    -- asignación desde una query con SELECT ... INTO
    SELECT COUNT(*) INTO total_clientes FROM northwind_dwh.dim_customer;

    INSERT INTO salida VALUES
        ('Motor',          nombre || ' ' || version),
        ('Total clientes', total_clientes::TEXT);
END
$$;

SELECT * FROM salida;

**Tipos basados en columnas existentes — `%TYPE` y `%ROWTYPE`:**

Útiles para evitar repetir tipos:

```sql
DECLARE
    nombre_cliente  northwind_dwh.dim_customer.company_name%TYPE;   -- mismo tipo que la columna
    fila_cliente    northwind_dwh.dim_customer%ROWTYPE;             -- una fila completa
```

Si después cambias el tipo de la columna, las variables se ajustan automáticamente.

## Bloques anidados

Dentro de un `BEGIN ... END` puedes abrir **otro** bloque, con su propio `DECLARE`. Sirve para dar **scope local** a una variable: la declarada en el sub-bloque solo existe ahí, y el sub-bloque sí ve las variables del bloque externo.

In [ ]:
%%sql
DROP TABLE IF EXISTS salida;
CREATE TEMP TABLE salida (donde TEXT, valor INTEGER);

DO $$
DECLARE
    x INTEGER := 1;                 -- variable del bloque externo
BEGIN
    INSERT INTO salida VALUES ('externo: x', x);

    -- sub-bloque anidado con su propia variable
    DECLARE
        y INTEGER := 100;           -- solo existe dentro de este sub-bloque
    BEGIN
        INSERT INTO salida VALUES ('interno: x + y', x + y);   -- ve la x de afuera
    END;

    -- aquí 'y' ya no existe; 'x' sigue disponible
    INSERT INTO salida VALUES ('externo: x otra vez', x);
END
$$;

SELECT * FROM salida;

El sub-bloque ve `x` (del bloque externo), pero `y` **solo vive dentro de él** — fuera del sub-bloque `y` ya no existe. Eso es el **scope**: cada bloque controla sus propias variables.

**El uso más común en la práctica: manejo local de excepciones.** Solo un sub-bloque puede tener su propia cláusula `EXCEPTION`, que **atrapa un error dentro de esa sección** sin abortar el bloque externo. El patrón clásico: procesar fila por fila y, si una falla, registrarla y **seguir con la siguiente** en vez de tumbar todo el proceso.

In [ ]:
%%sql
DROP TABLE IF EXISTS resultados;
CREATE TEMP TABLE resultados (entrada TEXT, estado TEXT, detalle TEXT);

DO $$
DECLARE
    fila RECORD;
    n    INTEGER;
BEGIN
    FOR fila IN SELECT v FROM (VALUES ('10'), ('20'), ('abc'), ('40')) AS t(v) LOOP
        BEGIN
            n := fila.v::INTEGER;        -- falla si v no es un número
            INSERT INTO resultados VALUES (fila.v, 'OK', 'convertido a ' || n);
        EXCEPTION WHEN others THEN
            -- si ESTA fila falla, la registramos y seguimos con la siguiente
            INSERT INTO resultados VALUES (fila.v, 'ERROR', SQLERRM);
        END;
    END LOOP;
END
$$;

SELECT * FROM resultados;

La fila `'abc'` falla al convertirse a entero, pero gracias al `EXCEPTION` del sub-bloque **el loop no se detiene**: la marca como `ERROR` (con el mensaje en `SQLERRM`) y continúa con `'40'`. Sin el sub-bloque interno, ese único error **abortaría todo**.

> Cada bloque con `EXCEPTION` crea por debajo un *savepoint* (sub-transacción), que tiene un costo — úsalo donde de verdad necesitas la recuperación, no en cada paso.

## Cierre

Lo que cubriste en este notebook:

| Tema | Construcción clave |
|---|---|
| Por qué PL/pgSQL | Lo que SQL puro no puede hacer: variables y control de flujo |
| Bloque anónimo | `DO $$ BEGIN ... END $$;` |
| Variables | `DECLARE nombre TIPO := valor;` + asignación con `:=` |
| Asignar desde query | `SELECT col INTO variable FROM ...` |
| Bloques anidados | `BEGIN` dentro de `BEGIN`: *scope* local y manejo de excepciones por sección |
| Ver la salida | El bloque escribe en una tabla temporal y un `SELECT` la muestra |

El siguiente notebook (**02 — Control de flujo**) profundiza en las construcciones procedurales: `IF`, `CASE`, `FOR`, `WHILE`.

---

<p align="center">
<a href="Readme.md">← Volver al índice del Tema 06</a> | <a href="02_control_de_flujo.ipynb">Siguiente: Notebook 02 — Control de flujo →</a>
</p>